# Lesson 04 · Equivalence of Categories

Isomorphic categories are rare; equivalent ones are everywhere. The right notion of sameness for categories is sameness up to natural isomorphism — and it comes with a three-part test.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-04-equivalence-of-categories.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §7 — Equivalences by hand

**`setup.py`** — carried over from Lessons 1 and 2

In [1]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar = set(objects), arrows
        self.comp, self.id = comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        # 1. each identity is typed X -> X
        for X in self.obj:
            assert self.ar[self.id[X]] == (X, X)
        # 2. every composable pair has a composite, correctly typed
        for g, f in product(self.ar, repeat=2):
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        # 3. identity law on both sides
        for f in self.ar:
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        # 4. associativity
        for h, g, f in product(self.ar, repeat=3):
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

class Functor:
    """F : C -> D, given by a map on objects and a map on arrows."""
    def __init__(self, C, D, on_obj, on_arr):
        self.C, self.D, self.on_obj, self.on_arr = C, D, on_obj, on_arr

    def check(self):
        C, D, Fo, Fa = self.C, self.D, self.on_obj, self.on_arr
        for f in C.ar:                        # typing: F f : F X -> F Y
            X, Y = C.ar[f]
            assert D.ar[Fa[f]] == (Fo[X], Fo[Y])
        for X in C.obj:                       # preserves identities
            assert Fa[C.id[X]] == D.id[Fo[X]]
        for g, f in product(C.ar, repeat=2):  # preserves composition
            if C.composable(g, f):
                assert Fa[C.comp[(g, f)]] == D.comp[(Fa[g], Fa[f])]
        return True

def arrows_between(C, X, Y):
    "all arrows X -> Y of the finite category C"
    return [a for a, (d, c) in C.ar.items() if d == X and c == Y]

def make_poset(objs, leq):
    "a finite preorder as a thin category; the arrow a -> b is named 'a<=b'"
    ar = {f"{a}<={b}": (a, b) for a in objs for b in objs if leq(a, b)}
    comp = {(g, f): f"{ar[f][0]}<={ar[g][1]}"
            for g in ar for f in ar if ar[f][1] == ar[g][0]}
    return FiniteCategory(objs, ar, comp, {a: f"{a}<={a}" for a in objs})

# the category 2 = ( 0 --u--> 1 )
two_ar   = {'i0':(0,0), 'i1':(1,1), 'u':(0,1)}
two_comp = {('i0','i0'):'i0', ('i1','i1'):'i1',
            ('u','i0'):'u',  ('i1','u'):'u'}
Two = FiniteCategory({0,1}, two_ar, two_comp, {0:'i0', 1:'i1'})

**`skeleton.py`** — isomorphism classes and a skeleton

In [2]:
def is_iso(C, f):
    X, Y = C.ar[f]
    return any(C.comp[(g, f)] == C.id[X] and C.comp[(f, g)] == C.id[Y]
               for g in arrows_between(C, Y, X))

def iso_classes(C):
    classes = []
    for X in sorted(C.obj, key=str):
        for cl in classes:
            if any(is_iso(C, f) for f in arrows_between(C, X, cl[0])):
                cl.append(X); break
        else:
            classes.append([X])
    return classes

def skeleton(C):
    "the full subcategory on one object from each isomorphism class"
    keep = {max(cl) for cl in iso_classes(C)}      # one object per class
    ar = {a: dc for a, dc in C.ar.items()
          if dc[0] in keep and dc[1] in keep}
    comp = {(g, f): C.comp[(g, f)]
            for g in ar for f in ar if ar[f][1] == ar[g][0]}
    return FiniteCategory(keep, ar, comp, {X: C.id[X] for X in keep})

divides = lambda a, b: b % a == 0
Signed = make_poset({1, -1, 2, -2, 3, -3, 6, -6}, divides)
Sk = skeleton(Signed)
print("classes :", iso_classes(Signed))
print("arrows  :", len(Signed.ar), "in the preorder,",
      len(Sk.ar), "in its skeleton", sorted(Sk.obj))
print("skeleton is a category:", Sk.check())

classes : [[-1, 1], [-2, 2], [-3, 3], [-6, 6]]
arrows  : 36 in the preorder, 9 in its skeleton [1, 2, 3, 6]
skeleton is a category: True


**`three_part_test.py`** — faithful, full, essentially surjective

In [3]:
def faithful(F):
    C = F.C
    return all(F.on_arr[f] != F.on_arr[g]
               for f in C.ar for g in C.ar
               if f != g and C.ar[f] == C.ar[g])

def full(F):
    C, D = F.C, F.D
    return all({F.on_arr[f] for f in arrows_between(C, X, Y)}
               == set(arrows_between(D, F.on_obj[X], F.on_obj[Y]))
               for X in C.obj for Y in C.obj)

def essentially_surjective(F):
    D = F.D
    reached = lambda Y: any(is_iso(D, k) for X in F.C.obj
                            for k in arrows_between(D, F.on_obj[X], Y))
    return all(reached(Y) for Y in D.obj)

def is_equivalence(F):
    return faithful(F) and full(F) and essentially_surjective(F)

# the walking isomorphism  I = ( a <--> b ),  and the one-object category
I_ar   = {'ia': ('a','a'), 'ib': ('b','b'), 'u': ('a','b'), 'v': ('b','a')}
I_comp = {('ia','ia'):'ia', ('ib','ib'):'ib',
          ('u','ia'):'u', ('ib','u'):'u', ('v','ib'):'v', ('ia','v'):'v',
          ('v','u'):'ia', ('u','v'):'ib'}
Iso = FiniteCategory({'a','b'}, I_ar, I_comp, {'a':'ia', 'b':'ib'})
One = FiniteCategory({'*'}, {'i': ('*','*')}, {('i','i'):'i'}, {'*':'i'})
assert Iso.check() and One.check()

pick_a  = Functor(One, Iso, {'*':'a'}, {'i':'ia'})
squash  = Functor(Two, One, {0:'*', 1:'*'}, {'i0':'i', 'i1':'i', 'u':'i'})
include = Functor(Sk, Signed, {X: X for X in Sk.obj}, {a: a for a in Sk.ar})

tests = [("1 -> I", pick_a), ("2 -> 1", squash), ("skeleton", include)]
for name, F in tests:
    assert F.check()
    print(f"{name:9}", "faithful:", faithful(F), " full:", full(F),
          " ess. surjective:", essentially_surjective(F),
          " => equivalence:", is_equivalence(F))

1 -> I    faithful: True  full: True  ess. surjective: True  => equivalence: True
2 -> 1    faithful: True  full: False  ess. surjective: True  => equivalence: False
skeleton  faithful: True  full: True  ess. surjective: True  => equivalence: True


## Exercise 14

**`divisors_of_8.py`**

In [4]:
Signed8 = make_poset({1, -1, 2, -2, 4, -4, 8, -8}, divides)
Sk8 = skeleton(Signed8)
print(iso_classes(Signed8))
print(len(Signed8.ar), "arrows;", len(Sk8.ar), "in the skeleton",
      sorted(Sk8.obj))

objs8 = Signed8.obj
absval = Functor(Signed8, Sk8, {n: abs(n) for n in objs8},
                 {f"{a}<={b}": f"{abs(a)}<={abs(b)}"
                  for a in objs8 for b in objs8 if divides(a, b)})
print(absval.check(), is_equivalence(absval))

[[-1, 1], [-2, 2], [-4, 4], [-8, 8]]
40 arrows; 10 in the skeleton [1, 2, 4, 8]
True True
